In [6]:
import os
import streamlit as st
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv

In [7]:
load_dotenv()

True

In [8]:
## load the GROQ API Key and openAI key
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")

In [9]:
# Function to load documents
def load_documents_from_directory(directory_path):
    documents = []
    for file in os.listdir(directory_path):
        path = os.path.join(directory_path, file)
        if file.endswith(".pdf"):
            loader = PyPDFLoader(path)
        elif file.endswith(".txt"):
            loader = TextLoader(path)
        else:
            continue
        documents.extend(loader.load())
    return documents

In [10]:
# Function to embed and save to vectorstore
def get_vectorstore(documents):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")
    return vectorstore


In [11]:
# Define summary and QA prompts
SUMMARY_PROMPT = PromptTemplate(
    input_variables=["context"],
    template="Summarize the following content:\n\n{context}"
)

QA_PROMPT = PromptTemplate(
    input_variables=["context", "input"],
    template="Answer the question based on the context:\n\nContext: {context}\n\nQuestion: {input}"
)


In [13]:
# Main App
st.title("📚 RAG Bot with Summarization and QA")

# Load documents
with st.spinner("Loading and embedding documents..."):
    docs = load_documents_from_directory("docs")
    vectorstore = get_vectorstore(docs)

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(model_name="gpt-4", temperature=0.7)

2025-06-23 12:04:45.832 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-23 12:04:45.835 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-23 12:04:45.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-23 12:04:45.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-23 12:04:45.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-23 12:04:46.347 Thread 'Thread-14': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-23 12:04:46.358 Thread 'Thread-14': missing ScriptRunContext! This warning can be ignored when running in bare mode.
C:\Users\Mathan\AppData\Local\Temp\ipykernel_20180\161237369.py:6: LangChainDeprecationWarning: The class `OpenAIEmbeddi

APIConnectionError: Connection error.

In [ ]:
# Summarization chain
summarization_chain = create_stuff_documents_chain(llm, SUMMARY_PROMPT)

# RAG QA Chain
rag_chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, QA_PROMPT))

# Sidebar controls
option = st.sidebar.radio("Choose a task:", ["Summarize All", "Ask a Question"])


In [ ]:
# Summary
if option == "Summarize All":
    with st.spinner("Generating summary..."):
        summary = summarization_chain.invoke({"input_documents": docs})
        st.subheader("📄 Summary")
        st.write(summary["output"])

# Q&A
else:
    query = st.text_input("Ask a question about the documents:")
    if query:
        with st.spinner("Thinking..."):
            result = rag_chain.invoke({"input": query})
            st.subheader("🤖 Answer")
            st.write(result["answer"])
